In [1]:
import pandas as pd
import numpy as np
import math
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

import ipywidgets as widgets
from IPython.display import display, clear_output

print("All imports loaded successfully.")


All imports loaded successfully.


In [2]:
# ── 30-Location Japan Tourism Dataset (Expanded Tags + Coordinates) ──────────
data = [
    # ═══════════════════════  MAJOR ANCHORS  ═══════════════════════
    {"region": "Tokyo", "is_anchor": True, "tourist_count": 200000, "capacity": 150000,
     "avg_delay_minutes": 25, "cultural_score": 85, "experiential_score": 95,
     "infrastructure_score": 95, "promotion_score": 100, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.6895, "lon": 139.6917,
     "tags": ["urban", "modern", "shopping", "food", "nightlife", "anime",
             "tech", "skyscraper", "street_food", "museum", "lively",
             "fashion", "train_hub", "neon", "pop_culture"]},

    {"region": "Kyoto", "is_anchor": True, "tourist_count": 120000, "capacity": 80000,
     "avg_delay_minutes": 30, "cultural_score": 95, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 95, "cleanliness_score": 72,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.0116, "lon": 135.7681,
     "tags": ["historic", "temple", "shrine", "cherry_blossom", "traditional",
             "unesco", "culture", "geisha", "gardens", "bamboo",
             "matcha", "photo_spot", "architecture", "autumn_leaves", "crafts"]},

    {"region": "Osaka", "is_anchor": True, "tourist_count": 150000, "capacity": 120000,
     "avg_delay_minutes": 20, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 90, "promotion_score": 90, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.6937, "lon": 135.5023,
     "tags": ["urban", "food", "nightlife", "modern", "street_food",
             "castle", "lively", "comedy", "shopping", "takoyaki",
             "neon", "canal", "entertainment"]},

    {"region": "Hakone", "is_anchor": True, "tourist_count": 60000, "capacity": 40000,
     "avg_delay_minutes": 35, "cultural_score": 85, "experiential_score": 90,
     "infrastructure_score": 80, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.2326, "lon": 139.1070,
     "tags": ["hot_spring", "nature", "mountain", "view", "lake",
             "ryokan", "cable_car", "volcanic", "serene", "art_museum",
             "scenic_train", "relaxation", "fuji_view"]},

    {"region": "Nara", "is_anchor": True, "tourist_count": 70000, "capacity": 50000,
     "avg_delay_minutes": 25, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 80, "promotion_score": 80, "cleanliness_score": 74,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 34.6851, "lon": 135.8048,
     "tags": ["historic", "temple", "nature", "deer", "unesco",
             "traditional", "park", "buddha", "ancient", "serene",
             "photo_spot", "culture", "architecture"]},

    {"region": "Hiroshima", "is_anchor": True, "tourist_count": 90000, "capacity": 70000,
     "avg_delay_minutes": 20, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 75, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.3853, "lon": 132.4553,
     "tags": ["historic", "memorial", "peace", "temple", "island",
             "unesco", "culture", "coastal", "tram", "okonomiyaki",
             "architecture", "museum", "solemn"]},

    {"region": "Kamakura", "is_anchor": True, "tourist_count": 55000, "capacity": 40000,
     "avg_delay_minutes": 28, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 80, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 35.3192, "lon": 139.5467,
     "tags": ["historic", "temple", "coastal", "buddha", "shrine",
             "hiking", "beach", "traditional", "serene", "photo_spot",
             "architecture", "crafts", "day_trip"]},

    {"region": "Nikko", "is_anchor": True, "tourist_count": 45000, "capacity": 35000,
     "avg_delay_minutes": 30, "cultural_score": 92, "experiential_score": 88,
     "infrastructure_score": 70, "promotion_score": 75, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 36.7500, "lon": 139.6000,
     "tags": ["historic", "shrine", "mountain", "nature", "unesco",
             "waterfall", "autumn_leaves", "ornate", "cedar", "lake",
             "hiking", "traditional", "architecture"]},

    {"region": "Sapporo", "is_anchor": True, "tourist_count": 80000, "capacity": 70000,
     "avg_delay_minutes": 15, "cultural_score": 75, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 85, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 43.0621, "lon": 141.3544,
     "tags": ["urban", "snow", "food", "festival", "beer",
             "ski", "ramen", "park", "modern", "winter_sport",
             "lively", "night_market", "scenic"]},

    {"region": "Fukuoka", "is_anchor": True, "tourist_count": 65000, "capacity": 60000,
     "avg_delay_minutes": 10, "cultural_score": 80, "experiential_score": 90,
     "infrastructure_score": 88, "promotion_score": 80, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.5904, "lon": 130.4017,
     "tags": ["urban", "food", "historic", "shopping", "ramen",
             "shrine", "night_market", "canal", "modern", "lively",
             "street_food", "port"]},

    # ═══════════════════════  ORBIT DESTINATIONS  ═══════════════════════
    {"region": "Kanazawa", "is_anchor": False, "tourist_count": 25000, "capacity": 50000,
     "avg_delay_minutes": 8, "cultural_score": 85, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.5613, "lon": 136.6562,
     "tags": ["historic", "cherry_blossom", "garden", "traditional", "crafts",
             "geisha", "samurai", "museum", "culture", "serene",
             "architecture", "matcha", "autumn_leaves"]},

    {"region": "Takayama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 80, "experiential_score": 75,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 90,
     "has_coach_parking": False, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 36.1461, "lon": 137.2517,
     "tags": ["alpine", "mountain", "rural", "historic", "traditional",
             "festival", "sake_brewery", "edo_town", "crafts", "serene",
             "hiking", "wood_carving", "morning_market"]},

    {"region": "Shirakawa-go", "is_anchor": False, "tourist_count": 22000, "capacity": 20000,
     "avg_delay_minutes": 15, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 50, "promotion_score": 70, "cleanliness_score": 80,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 36.2578, "lon": 136.9061,
     "tags": ["rural", "historic", "snow", "mountain", "unesco",
             "traditional", "thatched_roof", "village", "photo_spot", "serene",
             "farming", "culture", "winter"]},

    {"region": "Koyasan", "is_anchor": False, "tourist_count": 12000, "capacity": 25000,
     "avg_delay_minutes": 5, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 60, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 34.2130, "lon": 135.5830,
     "tags": ["temple", "historic", "mountain", "rural", "unesco",
             "buddhist", "cemetery", "serene", "meditation", "cable_car",
             "traditional", "spiritual", "pilgrim", "shojin_ryori"]},

    {"region": "Nagasaki", "is_anchor": False, "tourist_count": 20000, "capacity": 45000,
     "avg_delay_minutes": 8, "cultural_score": 88, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 32.7503, "lon": 129.8779,
     "tags": ["historic", "memorial", "coastal", "food", "tram",
             "culture", "church", "port", "island", "museum",
             "solemn", "international", "night_view"]},

    {"region": "Tottori", "is_anchor": False, "tourist_count": 12000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 70, "experiential_score": 78,
     "infrastructure_score": 60, "promotion_score": 40, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 35.5011, "lon": 134.2351,
     "tags": ["rural", "nature", "sand_dunes", "coastal", "adventure",
             "camel_ride", "paragliding", "museum", "quiet", "scenic",
             "seafood", "off_beaten_path"]},

    {"region": "Akita", "is_anchor": False, "tourist_count": 9000, "capacity": 30000,
     "avg_delay_minutes": 3, "cultural_score": 65, "experiential_score": 72,
     "infrastructure_score": 55, "promotion_score": 35, "cleanliness_score": 82,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 39.7186, "lon": 140.1024,
     "tags": ["rural", "nature", "festival", "mountain", "snow",
             "onsen", "lake", "samurai", "rice_paddy", "quiet",
             "traditional", "rustic"]},

    {"region": "Shimane", "is_anchor": False, "tourist_count": 7000, "capacity": 25000,
     "avg_delay_minutes": 2, "cultural_score": 75, "experiential_score": 70,
     "infrastructure_score": 50, "promotion_score": 30, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 35.4723, "lon": 133.0505,
     "tags": ["historic", "shrine", "garden", "rural", "mythology",
             "traditional", "serene", "castle", "quiet", "culture",
             "off_beaten_path", "sunset"]},

    {"region": "Okayama", "is_anchor": False, "tourist_count": 15000, "capacity": 40000,
     "avg_delay_minutes": 5, "cultural_score": 82, "experiential_score": 75,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.6551, "lon": 133.9195,
     "tags": ["historic", "garden", "castle", "culture", "traditional",
             "crafts", "denim", "fruit", "cycling", "serene",
             "architecture", "museum"]},

    {"region": "Kagawa", "is_anchor": False, "tourist_count": 14000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 82,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.3401, "lon": 134.0434,
     "tags": ["food", "coastal", "shrine", "udon", "island",
             "art", "garden", "pilgrimage", "scenic", "cycling",
             "bridge", "traditional"]},

    {"region": "Naoshima", "is_anchor": False, "tourist_count": 11000, "capacity": 15000,
     "avg_delay_minutes": 12, "cultural_score": 88, "experiential_score": 90,
     "infrastructure_score": 55, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 34.4614, "lon": 133.9958,
     "tags": ["art", "coastal", "modern", "rural", "museum",
             "architecture", "island", "sculpture", "gallery", "photo_spot",
             "minimalist", "design", "cycling"]},

    {"region": "Matsuyama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 85, "experiential_score": 82,
     "infrastructure_score": 72, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.8392, "lon": 132.7657,
     "tags": ["hot_spring", "historic", "castle", "tram", "traditional",
             "literature", "garden", "relaxation", "food", "serene",
             "architecture", "culture"]},

    {"region": "Beppu", "is_anchor": False, "tourist_count": 25000, "capacity": 45000,
     "avg_delay_minutes": 10, "cultural_score": 75, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.2846, "lon": 131.4914,
     "tags": ["hot_spring", "nature", "coastal", "volcanic", "steam",
             "relaxation", "ryokan", "sand_bath", "scenic", "food",
             "hell_tour", "unique"]},

    {"region": "Kurokawa Onsen", "is_anchor": False, "tourist_count": 8000, "capacity": 12000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 45, "promotion_score": 40, "cleanliness_score": 92,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 33.1283, "lon": 131.0714,
     "tags": ["hot_spring", "rural", "mountain", "ryokan", "serene",
             "traditional", "nature", "relaxation", "lantern", "rustic",
             "intimate", "off_beaten_path"]},

    {"region": "Kagoshima", "is_anchor": False, "tourist_count": 16000, "capacity": 38000,
     "avg_delay_minutes": 5, "cultural_score": 78, "experiential_score": 85,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 83,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 31.5966, "lon": 130.5571,
     "tags": ["nature", "coastal", "mountain", "hot_spring", "volcanic",
             "scenic", "food", "garden", "historic", "tram",
             "island", "adventure", "subtropical"]},

    {"region": "Sendai", "is_anchor": False, "tourist_count": 30000, "capacity": 65000,
     "avg_delay_minutes": 8, "cultural_score": 82, "experiential_score": 80,
     "infrastructure_score": 85, "promotion_score": 60, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 38.2682, "lon": 140.8694,
     "tags": ["urban", "historic", "food", "festival", "castle",
             "shopping", "beef_tongue", "tanabata", "park", "modern",
             "day_trip", "lively"]},

    {"region": "Aomori", "is_anchor": False, "tourist_count": 10000, "capacity": 35000,
     "avg_delay_minutes": 3, "cultural_score": 75, "experiential_score": 80,
     "infrastructure_score": 60, "promotion_score": 45, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 40.8244, "lon": 140.7400,
     "tags": ["nature", "snow", "rural", "festival", "nebuta",
             "apple", "hiking", "mountain", "lake", "scenic",
             "autumn_leaves", "quiet", "rustic"]},

    {"region": "Nagano", "is_anchor": False, "tourist_count": 22000, "capacity": 50000,
     "avg_delay_minutes": 7, "cultural_score": 85, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 60, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.6513, "lon": 138.1810,
     "tags": ["mountain", "snow", "nature", "temple", "ski",
             "monkey_park", "hiking", "alpine", "traditional", "soba",
             "scenic", "onsen", "winter_sport"]},

    {"region": "Matsumoto", "is_anchor": False, "tourist_count": 15000, "capacity": 35000,
     "avg_delay_minutes": 5, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 55, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.2380, "lon": 137.9720,
     "tags": ["castle", "historic", "alpine", "traditional", "culture",
             "crafts", "soba", "museum", "architecture", "serene",
             "mountain", "wasabi", "art"]},

    {"region": "Toyama", "is_anchor": False, "tourist_count": 13000, "capacity": 40000,
     "avg_delay_minutes": 4, "cultural_score": 75, "experiential_score": 85,
     "infrastructure_score": 68, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.6953, "lon": 137.2114,
     "tags": ["alpine", "nature", "coastal", "snow", "scenic",
             "glass_art", "sushi", "firefly_squid", "mountain", "dam",
             "hiking", "modern", "quiet"]},
]

df = pd.DataFrame(data)

# Quick validation
tag_counts = df["tags"].apply(len)
print(f"Locations: {len(df)}  |  Anchors: {df['is_anchor'].sum()}  |  Orbits: {(~df['is_anchor']).sum()}")
print(f"Tags per location - min: {tag_counts.min()}, max: {tag_counts.max()}, mean: {tag_counts.mean():.1f}")
print(f"Coordinates present: {df['lat'].notna().all() and df['lon'].notna().all()}")
df[["region", "is_anchor", "lat", "lon", "cultural_score"]].head(5)


Locations: 30  |  Anchors: 10  |  Orbits: 20
Tags per location - min: 12, max: 15, mean: 12.8
Coordinates present: True


,region,is_anchor,lat,lon,cultural_score
0,Tokyo,True,35.6895,139.6917,85
1,Kyoto,True,35.0116,135.7681,95
2,Osaka,True,34.6937,135.5023,80
3,Hakone,True,35.2326,139.1070,85
4,Nara,True,34.6851,135.8048,95


In [3]:
# ── Phase 1 Congestion Helpers ────────────────────────────────────────────────
def piecewise_load_score(ratio: float) -> float:
    """Piecewise congestion score with no ceiling above ratio >= 1.2."""
    if ratio < 0.3:
        return ratio * (20 / 0.3)
    elif ratio < 0.7:
        return 20 + (ratio - 0.3) * (30 / 0.4)
    elif ratio < 0.9:
        return 50 + (ratio - 0.7) * (25 / 0.2)
    elif ratio < 1.0:
        return 75 + (ratio - 0.9) * (15 / 0.1)
    elif ratio < 1.2:
        return 90 + (ratio - 1.0) * (10 / 0.2)
    else:
        return 100 + ((ratio - 1.2) * 20)


def compute_congestion_index(load_score: float, delay_score: float) -> float:
    """Dominance Rule: delay amplifies when load_score >= 100."""
    if load_score >= 100:
        return load_score + (delay_score * 0.20)
    return (load_score * 0.70) + (delay_score * 0.30)


def congestion_flag(index: float) -> str:
    if index < 20:   return "Severely undertouristed"
    elif index < 40: return "Undertouristed"
    elif index < 75: return "Balanced"
    elif index < 100: return "Approaching overtourism"
    else:             return "Severely overtouristed"


def compute_dynamic_penalty(row: pd.Series, group_size: int) -> int:
    """Group-size-gated feasibility penalty."""
    if group_size >= 15:
        return (
            (0 if row["has_coach_parking"]  else 20) +
            (0 if row["group_dining_nearby"] else 15) +
            (10 if row["requires_long_walk"] else 0)
        )
    else:
        return (5 if row["requires_long_walk"] else 0)


# ── Flaw 3 Fix: Piecewise Congestion Penalty ─────────────────────────────────
def congestion_penalty(index: float) -> float:
    """Asymmetric congestion penalty — mild for undertouristed, strong for overtouristed."""
    if index < 20:
        return index * 0.10
    elif index < 75:
        return index * 0.30
    else:
        return index * 0.50


# ── Phase 3: A* Spatial Graph ─────────────────────────────────────────────────
def build_japan_astar_graph(df: pd.DataFrame) -> nx.Graph:
    """
    Build a fully-connected weighted graph of Japan's transit network.
    Edge weights represent realistic travel times in minutes.
    """
    G = nx.Graph()

    # Add nodes with lat/lon attributes
    for _, row in df.iterrows():
        G.add_node(row["region"], lat=row["lat"], lon=row["lon"])

    # ── Tokaido-Sanyo Shinkansen Spine ────────────────────────────────────
    G.add_edge("Tokyo",      "Hakone",     weight=40)
    G.add_edge("Hakone",     "Kyoto",      weight=120)
    G.add_edge("Kyoto",      "Osaka",      weight=15)
    G.add_edge("Osaka",      "Nara",       weight=40)
    G.add_edge("Osaka",      "Hiroshima",  weight=90)
    G.add_edge("Hiroshima",  "Fukuoka",    weight=65)

    # ── Kanto Branches ────────────────────────────────────────────────────
    G.add_edge("Tokyo",      "Kamakura",   weight=55)
    G.add_edge("Tokyo",      "Nikko",      weight=110)

    # ── Tohoku Shinkansen ─────────────────────────────────────────────────
    G.add_edge("Tokyo",      "Sendai",     weight=90)
    G.add_edge("Sendai",     "Aomori",     weight=100)
    G.add_edge("Sendai",     "Akita",      weight=110)

    # ── Hokkaido ──────────────────────────────────────────────────────────
    G.add_edge("Aomori",     "Sapporo",    weight=300)

    # ── Hokuriku / Central Japan ──────────────────────────────────────────
    G.add_edge("Kyoto",      "Kanazawa",   weight=130)
    G.add_edge("Kanazawa",   "Toyama",     weight=25)
    G.add_edge("Kanazawa",   "Shirakawa-go", weight=60)
    G.add_edge("Shirakawa-go", "Takayama", weight=50)
    G.add_edge("Tokyo",      "Nagano",     weight=90)
    G.add_edge("Nagano",     "Matsumoto",  weight=50)
    G.add_edge("Matsumoto",  "Takayama",   weight=120)

    # ── Kansai Branches ───────────────────────────────────────────────────
    G.add_edge("Osaka",      "Koyasan",    weight=90)

    # ── San'yo / San'in ───────────────────────────────────────────────────
    G.add_edge("Hiroshima",  "Okayama",    weight=40)
    G.add_edge("Okayama",    "Tottori",    weight=120)
    G.add_edge("Tottori",    "Shimane",    weight=120)

    # ── Shikoku ───────────────────────────────────────────────────────────
    G.add_edge("Okayama",    "Kagawa",     weight=55)
    G.add_edge("Kagawa",     "Naoshima",   weight=30)
    G.add_edge("Kagawa",     "Matsuyama",  weight=150)

    # ── Kyushu ────────────────────────────────────────────────────────────
    G.add_edge("Fukuoka",    "Nagasaki",   weight=110)
    G.add_edge("Fukuoka",    "Beppu",      weight=120)
    G.add_edge("Fukuoka",    "Kagoshima",  weight=75)
    G.add_edge("Beppu",      "Kurokawa Onsen", weight=90)

    return G


def haversine_heuristic(u, v, d) -> float:
    """
    A* admissible heuristic: Haversine great-circle distance
    converted to an optimistic time estimate at Shinkansen speed (300 km/h).
    """
    lat1, lon1 = d[u]["lat"], d[u]["lon"]
    lat2, lon2 = d[v]["lat"], d[v]["lon"]

    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance_km = R * c

    heuristic_time = (distance_km / 300.0) * 60.0
    return heuristic_time


def get_astar_commute_minutes(graph: nx.Graph, anchor: str, orbit: str) -> float:
    """Return A* shortest-path travel time (minutes) between anchor and orbit."""
    try:
        return nx.astar_path_length(
            graph,
            source=anchor,
            target=orbit,
            heuristic=lambda u, v: haversine_heuristic(u, v, graph.nodes),
            weight="weight",
        )
    except nx.NetworkXNoPath:
        return float("inf")


print("Phase 1 congestion helpers + Phase 3 A* spatial graph loaded.")


Phase 1 congestion helpers + Phase 3 A* spatial graph loaded.


In [4]:
# ── Two-Stage NLP + A* Spatial Graph Recommendation Engine ─────────────────────

def _compute_tfidf_scores(df: pd.DataFrame, anchor_name: str) -> dict:
    """Stage 1: TF-IDF cosine similarity (anchor vs all). Returns {region: score}."""
    tag_strings = df["tags"].apply(lambda t: " ".join(t))
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(tag_strings)
    anchor_idx = df.index[df["region"] == anchor_name][0]
    cos_sim = cosine_similarity(tfidf_matrix[anchor_idx], tfidf_matrix).flatten()
    return dict(zip(df["region"], np.round(cos_sim * 100, 2)))


def _compute_knn_profile_scores(df: pd.DataFrame, anchor_name: str,
                                 survivor_regions: list) -> dict:
    """Stage 2: KNN cosine-distance similarity on numerical features.
    Only scores survivors from Stage 1. Returns {region: score}."""
    num_cols = ["cultural_score", "experiential_score",
               "cleanliness_score", "infrastructure_score"]
    scaler = MinMaxScaler()
    num_matrix = scaler.fit_transform(df[num_cols].values)

    knn = NearestNeighbors(n_neighbors=len(df), metric="cosine")
    knn.fit(num_matrix)

    anchor_idx = df.index[df["region"] == anchor_name][0]
    distances, indices = knn.kneighbors(num_matrix[anchor_idx].reshape(1, -1))

    sim_map = {}
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        region = df.iloc[idx]["region"]
        if region in survivor_regions:
            sim_map[region] = round((1 - dist) * 100, 2)
    return sim_map


# Column schema for the output DataFrame
_OUTPUT_COLS = [
    "region", "tfidf_score", "knn_profile_score",
    "congestion_index", "congestion_flag",
    "dynamic_penalty", "commute_minutes",
    "total_penalty", "final_match_score",
]


def generate_recommendations(
    df: pd.DataFrame,
    anchor_region: str,
    group_size: int,
    tourist_type: str,
) -> pd.DataFrame:
    """
    Two-Stage NLP Recommendation Engine with A* commute (display only).

    Commute minutes are calculated via A* on the transit graph but do NOT
    penalise the final score — they are provided as informational context.
    """
    tourist_type = tourist_type.lower().strip()
    if tourist_type not in ("international", "native"):
        raise ValueError("tourist_type must be 'international' or 'native'")

    anchor_row = df.loc[df["region"] == anchor_region]
    if anchor_row.empty:
        raise ValueError(f"Anchor region '{anchor_region}' not found.")

    # ── Build A* transit graph ─────────────────────────────────────────────
    transit_graph = build_japan_astar_graph(df)

    # ── Stage 1: TF-IDF cosine similarity ─────────────────────────────────
    tfidf_scores = _compute_tfidf_scores(df, anchor_region)

    orbits = df[df["region"] != anchor_region].copy()
    orbits["tfidf_score"] = orbits["region"].map(tfidf_scores)
    survivors = orbits[orbits["tfidf_score"] > 0.0].copy()

    if survivors.empty:
        return pd.DataFrame(columns=_OUTPUT_COLS)

    # ── FIX 1: Min-max renormalise TF-IDF within survivor set ─────────────
    tmin = survivors["tfidf_score"].min()
    tmax = survivors["tfidf_score"].max()
    if tmax > tmin:
        survivors["tfidf_score"] = (
            (survivors["tfidf_score"] - tmin) / (tmax - tmin) * 100
        ).round(2)
    else:
        survivors["tfidf_score"] = 75.0

    # ── Stage 2: KNN profile re-ranking ───────────────────────────────────
    survivor_list = survivors["region"].tolist()
    knn_scores = _compute_knn_profile_scores(df, anchor_region, survivor_list)
    survivors["knn_profile_score"] = survivors["region"].map(knn_scores)

    # ── Score each survivor ───────────────────────────────────────────────
    results = []
    for _, row in survivors.iterrows():

        # ── Phase 3: A* commute (display only, no penalty) ────────────────
        commute = get_astar_commute_minutes(
            transit_graph, anchor_region, row["region"]
        )

        # ── Attractiveness weights ────────────────────────────────────────
        if tourist_type == "international":
            attractiveness = (
                row["cultural_score"]      * 0.40 +
                row["cleanliness_score"]   * 0.30 +
                row["experiential_score"]  * 0.20 +
                row["infrastructure_score"]* 0.10
            )
        else:
            attractiveness = (
                row["experiential_score"]  * 0.40 +
                row["infrastructure_score"]* 0.30 +
                row["cleanliness_score"]   * 0.20 +
                row["cultural_score"]      * 0.10
            )

        # ── Base score ────────────────────────────────────────────────────
        base_score = (
            row["tfidf_score"]        * 0.50 +
            row["knn_profile_score"]  * 0.30 +
            attractiveness            * 0.20
        )

        # ── Congestion ────────────────────────────────────────────────────
        load_ratio  = row["tourist_count"] / row["capacity"]
        load_score  = piecewise_load_score(load_ratio)
        delay_score = min((row["avg_delay_minutes"] / 45) * 100, 100)
        c_index     = compute_congestion_index(load_score, delay_score)
        c_flag      = congestion_flag(c_index)

        # ── FIX 3: Piecewise congestion penalty ──────────────────────────
        c_penalty = congestion_penalty(c_index)

        # ── Feasibility penalty ───────────────────────────────────────────
        d_penalty = compute_dynamic_penalty(row, group_size)

        # ── FIX 2: Capped total penalty (no distance penalty) ─────────────
        total_raw = round(c_penalty + d_penalty, 2)
        total_capped = round(min(total_raw, 0.65 * base_score), 2)

        final = round(base_score - total_capped, 2)

        results.append({
            "region":             row["region"],
            "tfidf_score":        row["tfidf_score"],
            "knn_profile_score":  row["knn_profile_score"],
            "congestion_index":   round(c_index, 2),
            "congestion_flag":    c_flag,
            "dynamic_penalty":    d_penalty,
            "commute_minutes":    round(commute),
            "total_penalty":      total_capped,
            "final_match_score":  final,
        })

    if not results:
        return pd.DataFrame(columns=_OUTPUT_COLS)

    return (
        pd.DataFrame(results)
        .sort_values("final_match_score", ascending=False)
        .reset_index(drop=True)
    )


print("Recommendation engine loaded (TF-IDF + KNN + A* commute display).")


Recommendation engine loaded (TF-IDF + KNN + A* commute display).


In [5]:
from IPython.display import HTML

# ── Scenario 1: Kyoto | group=16 | international ─────────────────────────────
print("═" * 80)
print("SCENARIO 1: Kyoto | Coach (n=16) | International")
print("═" * 80)
s1 = generate_recommendations(df, "Kyoto", 16, "international")
if not s1.empty:
    diag1 = s1[["region", "tfidf_score", "knn_profile_score",
               "congestion_flag", "dynamic_penalty", "commute_minutes",
               "total_penalty", "final_match_score"]]
    display(diag1.style
           .format({"tfidf_score": "{:.2f}", "knn_profile_score": "{:.2f}",
                    "total_penalty": "{:.2f}", "final_match_score": "{:.2f}"})
           .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
           .background_gradient(subset=["commute_minutes"], cmap="YlOrRd")
           .set_caption("Scenario 1 — Kyoto | Coach | International"))
else:
    print("  ✗ No results!")

# ── Scenario 2: Kyoto | group=5 | international ──────────────────────────────
print("\n" + "═" * 80)
print("SCENARIO 2: Kyoto | FIT (n=5) | International")
print("═" * 80)
s2 = generate_recommendations(df, "Kyoto", 5, "international")
if not s2.empty:
    print(f"  ✓ Survivors: {len(s2)}")
    negs = s2[s2["final_match_score"] < 0]
    print(f"  ✓ No negative scores: {len(negs) == 0}")
    display(s2[["region", "tfidf_score", "commute_minutes",
               "total_penalty", "final_match_score"]].style
           .format({"tfidf_score": "{:.2f}", "total_penalty": "{:.2f}",
                    "final_match_score": "{:.2f}"})
           .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
           .set_caption("Scenario 2 — Kyoto | FIT | International"))
else:
    print("  ✗ No results!")

# ── Scenario 3: Tokyo | group=20 | native ────────────────────────────────────
print("\n" + "═" * 80)
print("SCENARIO 3: Tokyo | Coach (n=20) | Native")
print("═" * 80)
s3 = generate_recommendations(df, "Tokyo", 20, "native")
if not s3.empty:
    print(f"  ✓ Survivors: {len(s3)}")
    display(s3[["region", "commute_minutes", "total_penalty",
               "final_match_score"]].style
           .format({"total_penalty": "{:.2f}", "final_match_score": "{:.2f}"})
           .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
           .set_caption("Scenario 3 — Tokyo | Coach | Native"))
else:
    print("  ✗ No results!")

# ── Scenario 4: Nara | group=3 | native ──────────────────────────────────────
print("\n" + "═" * 80)
print("SCENARIO 4: Nara | FIT (n=3) | Native")
print("═" * 80)
s4 = generate_recommendations(df, "Nara", 3, "native")
if not s4.empty:
    print(f"  ✓ Survivors: {len(s4)}")
    print(f"  ✓ No negative scores: {(s4['final_match_score'] >= 0).all()}")
else:
    print("  ✗ No results!")


════════════════════════════════════════════════════════════════════════════════
SCENARIO 1: Kyoto | Coach (n=16) | International
════════════════════════════════════════════════════════════════════════════════


,region,tfidf_score,knn_profile_score,congestion_flag,dynamic_penalty,commute_minutes,total_penalty,final_match_score
0,Kanazawa,100.00,75.12,Undertouristed,0,130,8.95,80.27
1,Matsumoto,30.68,76.93,Undertouristed,0,300,7.22,48.12
2,Okayama,31.20,71.11,Undertouristed,0,145,6.38,46.67
3,Matsuyama,21.25,83.32,Undertouristed,0,350,7.76,44.48
4,Nagasaki,6.00,91.62,Undertouristed,0,280,8.07,39.28
5,Nagano,10.05,83.09,Undertouristed,0,250,7.80,39.19
6,Kagawa,10.32,79.72,Undertouristed,0,200,6.58,38.52
7,Kagoshima,0.20,82.92,Undertouristed,0,245,7.11,33.89
8,Sendai,0.00,83.08,Undertouristed,0,250,8.34,33.08
9,Fukuoka,10.19,88.30,Balanced,0,170,21.77,26.50



════════════════════════════════════════════════════════════════════════════════
SCENARIO 2: Kyoto | FIT (n=5) | International
════════════════════════════════════════════════════════════════════════════════
  ✓ Survivors: 22
  ✓ No negative scores: True


,region,tfidf_score,commute_minutes,total_penalty,final_match_score
0,Kanazawa,100.00,130,8.95,80.27
1,Matsumoto,30.68,300,7.22,48.12
2,Okayama,31.20,145,6.38,46.67
3,Matsuyama,21.25,350,7.76,44.48
4,Koyasan,20.88,105,13.03,39.68
5,Nagasaki,6.00,280,8.07,39.28
6,Nagano,10.05,250,7.80,39.19
7,Kagawa,10.32,200,6.58,38.52
8,Aomori,9.05,350,1.53,37.79
9,Shimane,23.30,385,1.44,35.16



════════════════════════════════════════════════════════════════════════════════
SCENARIO 3: Tokyo | Coach (n=20) | Native
════════════════════════════════════════════════════════════════════════════════
  ✓ Survivors: 16


,region,commute_minutes,total_penalty,final_match_score
0,Sendai,90,8.34,62.62
1,Fukuoka,330,21.77,61.31
2,Toyama,315,1.80,42.55
3,Osaka,175,54.94,42.20
4,Nagasaki,440,8.07,41.16
5,Kagoshima,405,7.11,37.72
6,Matsuyama,510,7.76,36.31
7,Kagawa,360,6.58,36.06
8,Beppu,450,10.22,35.53
9,Matsumoto,140,7.22,35.31



════════════════════════════════════════════════════════════════════════════════
SCENARIO 4: Nara | FIT (n=3) | Native
════════════════════════════════════════════════════════════════════════════════
  ✓ Survivors: 27
  ✓ No negative scores: True


In [ ]:
# ── Interactive Dashboard (A* Spatial Graph Engine) ────────────────────────────
out = widgets.Output()

anchor_dropdown = widgets.Dropdown(
    options=sorted(df["region"].unique().tolist()),
    value="Kyoto",
    description="Anchor:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

group_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description="Group Size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

type_dropdown = widgets.Dropdown(
    options=["international", "native"],
    value="international",
    description="Tourist Type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)


def render_recommendations(anchor_region, group_size, tourist_type):
    results = generate_recommendations(
        df=df, anchor_region=anchor_region,
        group_size=group_size, tourist_type=tourist_type,
    )

    DISPLAY = [
        "region", "tfidf_score", "knn_profile_score",
        "congestion_flag", "dynamic_penalty",
        "commute_minutes", "total_penalty", "final_match_score",
    ]

    with out:
        clear_output(wait=True)
        if results.empty:
            print("No thematically relevant orbits found.")
            return

        styled = (
            results[DISPLAY].style
            .format({
                "tfidf_score":       "{:.2f}",
                "knn_profile_score": "{:.2f}",
                "total_penalty":     "{:.2f}",
                "final_match_score": "{:.2f}",
            })
            .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
            .background_gradient(subset=["tfidf_score"],       cmap="Blues")
            .background_gradient(subset=["knn_profile_score"], cmap="PuBuGn")
            .background_gradient(subset=["commute_minutes"],   cmap="YlOrRd")
            .map(
                lambda v: "color: crimson; font-weight: bold"
                    if v == "Severely overtouristed"
                else ("color: darkorange; font-weight: bold"
                    if v == "Approaching overtourism" else ""),
                subset=["congestion_flag"],
            )
            .set_caption(
                f"DSS Recommendations | Anchor: {anchor_region} | "
                f"Group: {group_size} | Type: {tourist_type.capitalize()}"
            )
            .set_table_styles([{"selector": "", "props": [("width", "100%")]}])
        )
        display(HTML(
            '<div style="overflow-x:auto; max-width:100%">'
            + styled.to_html()
            + "</div>"
        ))


def _on_change(_):
    render_recommendations(
        anchor_dropdown.value,
        group_slider.value,
        type_dropdown.value,
    )


anchor_dropdown.observe(_on_change, names="value")
group_slider.observe(_on_change,    names="value")
type_dropdown.observe(_on_change,   names="value")

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>DSS Dashboard — A* Spatial Graph Engine</h3>"),
    widgets.HBox([anchor_dropdown, type_dropdown]),
    group_slider,
    widgets.HTML(
        "<small style='color:grey'>TF-IDF renormalised within survivors. "
        "Penalty capped at 65% of base score. "
        "Commute via A* on transit graph (display only, no score penalty). "
        "Group ≥ 15 → coach feasibility penalties.</small>"
    ),
])

display(controls, out)

# Trigger initial render
render_recommendations(
    anchor_dropdown.value,
    group_slider.value,
    type_dropdown.value,
)


Output()